# Singular Value Decomposition

This notebook uses the local NumPy implementation and keeps the factor order `A = U @ diag(S) @ V.T`.

In [ ]:
from pathlib import Path
import sys
candidates = (Path.cwd() / 'code', Path.cwd().parents[1] / 'code', Path.cwd() / 'phases/01-math-foundations/11-singular-value-decomposition/code')
code_dir = next(path for path in candidates if (path / 'svd.py').is_file())
sys.path.insert(0, str(code_dir))
from svd import compression_ratio, pseudoinverse_via_svd, reconstruct, svd_from_scratch, truncated_svd
import numpy as np


## Build It: power iteration on `A.T @ A`

The scratch routine returns `U`, singular values `S`, and `V` (not `V.T`). Reconstruct with `V.T`; signs may differ from another valid SVD.

In [ ]:
np.random.seed(11)
A = np.array([[3.0, 1.0], [1.0, 3.0], [2.0, -1.0]])
U, S, V = svd_from_scratch(A)
U.shape, S.shape, V.shape, np.round(S, 5), round(np.linalg.norm(A - reconstruct(U, S, V.T)), 8)


## Use It: truncation and least squares

`truncated_svd` provides the NumPy reference factors. The pseudoinverse keeps reciprocals only for singular values above its tolerance, which also handles rank-deficient systems.

In [ ]:
U2, S2, Vt2 = truncated_svd(np.diag([5.0, 2.0, 1.0]), 2)
A_tilde = reconstruct(U2, S2, Vt2)
A = np.array([[1.0, 1.0], [2.0, 1.0], [3.0, 1.0]])
b = np.array([3.0, 5.0, 6.0])
x = pseudoinverse_via_svd(A) @ b
A_tilde, np.round(x, 6), compression_ratio(100, 80, 5)


## Exercise

Compare rank 1 and rank 2 reconstructions of `diag([5,2,1])`. Then verify `A.T @ (A @ x - b)` is close to zero for the overdetermined fixture. Explain why PCA directions are the right singular vectors after centering.